# VAE Conversion

Encode raw Mario episodes with `stabilityai/sd-vae-ft-mse`. Each encoded episode contains deterministic VAE latents, the original step records, and encoding metadata.

```text
data/encoded_data/episodes/ep_000001/
  latents.npy   # float16 [T, 4, H/8, W/8]
  steps.jsonl  # copied from the raw episode
  meta.json    # raw metadata plus encoding details
```


In [ ]:
from __future__ import annotations

import json
import shutil
from dataclasses import asdict, dataclass
from pathlib import Path

import imageio.v2 as imageio
import numpy as np
import torch
from diffusers import AutoencoderKL

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "data-collection" else Path.cwd()
RAW_EPISODES_DIR = REPO_ROOT / "data" / "raw" / "episodes"
ENCODED_EPISODES_DIR = REPO_ROOT / "data" / "encoded_data" / "episodes"

print(f"Raw episodes: {RAW_EPISODES_DIR}")
print(f"Encoded episodes: {ENCODED_EPISODES_DIR}")


In [ ]:
@dataclass
class EncodingConfig:
    vae_model: str = "stabilityai/sd-vae-ft-mse"
    batch_size: int = 16
    latent_dtype: str = "float16"


CONFIG = EncodingConfig()


def select_device() -> str:
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


def load_vae(config: EncodingConfig):
    device = select_device()
    dtype = torch.float16 if device != "cpu" else torch.float32
    vae = AutoencoderKL.from_pretrained(config.vae_model)
    vae.to(device=device, dtype=dtype).eval()
    return vae, device, dtype


In [ ]:
def list_raw_episodes() -> list[Path]:
    return sorted(path for path in RAW_EPISODES_DIR.glob("ep_*") if (path / "frames.mkv").exists())


def frame_batch_to_tensor(frames: list[np.ndarray], device: str, dtype: torch.dtype) -> torch.Tensor:
    batch = np.stack([np.asarray(frame)[..., :3] for frame in frames])
    tensor = torch.from_numpy(batch).permute(0, 3, 1, 2).to(device=device, dtype=dtype)
    return tensor.div(127.5).sub(1.0)


@torch.inference_mode()
def encode_episode(raw_dir: Path, vae, device: str, dtype: torch.dtype, config: EncodingConfig) -> dict:
    encoded_dir = ENCODED_EPISODES_DIR / raw_dir.name
    tmp_dir = encoded_dir.with_name(encoded_dir.name + ".tmp")
    if encoded_dir.exists():
        shutil.rmtree(encoded_dir)
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True)

    latents = []
    with imageio.get_reader(raw_dir / "frames.mkv") as reader:
        batch = []
        for frame in reader:
            batch.append(frame)
            if len(batch) == config.batch_size:
                tensor = frame_batch_to_tensor(batch, device, dtype)
                latent = vae.encode(tensor).latent_dist.mean * vae.config.scaling_factor
                latents.append(latent.float().cpu().numpy().astype(config.latent_dtype))
                batch.clear()
        if batch:
            tensor = frame_batch_to_tensor(batch, device, dtype)
            latent = vae.encode(tensor).latent_dist.mean * vae.config.scaling_factor
            latents.append(latent.float().cpu().numpy().astype(config.latent_dtype))

    latent_array = np.concatenate(latents, axis=0)
    np.save(tmp_dir / "latents.npy", latent_array)
    shutil.copy2(raw_dir / "steps.jsonl", tmp_dir / "steps.jsonl")

    raw_metadata = json.loads((raw_dir / "meta.json").read_text())
    metadata = {
        **raw_metadata,
        "encoding": {
            **asdict(config),
            "latent_shape": list(latent_array.shape),
            "vae_scaling_factor": float(vae.config.scaling_factor),
            "latent_distribution": "mean",
        },
    }
    (tmp_dir / "meta.json").write_text(json.dumps(metadata, indent=2) + "\n")
    tmp_dir.rename(encoded_dir)
    return {"episode": raw_dir.name, "latent_shape": latent_array.shape}


def convert_dataset(config: EncodingConfig = CONFIG, dry_run: bool = False) -> list[dict]:
    episodes = list_raw_episodes()
    if dry_run:
        return [{"episode": episode.name, "status": "planned"} for episode in episodes]

    vae, device, dtype = load_vae(config)
    print(f"Encoding {len(episodes)} episodes on {device} with {dtype}.")
    summaries = []
    for index, episode in enumerate(episodes, start=1):
        summary = encode_episode(episode, vae, device, dtype, config)
        summaries.append(summary)
        print(f"[{index}/{len(episodes)}] {summary['episode']}: {summary['latent_shape']}")
    return summaries


In [ ]:
# Convert every raw episode. Re-running this cell replaces encoded episode directories.
summaries = convert_dataset()
print(f"Encoded episodes: {len(summaries)}")
summaries[:5]
